## 1. Data Sources

- 6_Damage_Sites_Aleppo_SDA.shp — UNOSAT point-level conflict damage assessment (event code CE20130604SYR, Syrian Civil War, 2012–2016), 35,936 points citywide. Key columns: SiteID, DmgCls_4, SensDt_4, GrpDmgCls, FldValid, Notes, Neighbrhd, StlmtNme, EventCode. CRS: EPSG:4326.
- 7_Aleppo_PercentageDamage_Neighborhood.shp — 127 named neighborhood/district polygons with a Percent field, Name and Name_Ar. CRS: EPSG:4326. Includes Old City quarters such as Aleppo Citadel, Al Jalloum, al Aqabeh, Farafira, Bayadah, al Kallasah, As-Sukkari, Bab Alfaraj, and others.
- 8_Aleppo_ResidentialPercentageDamage.shp — 553,296-polygon rasterized damage-density surface (ID, GRIDCODE 0–100). Not used for node/edge construction; it is included only as an optional visual backdrop.

## 2. Load and Filter Damage Sites

In [6]:
import warnings
import os
import pandas as pd
import networkx as nx
import geopandas as gpd
from pathlib import Path

warnings.filterwarnings("ignore")

# Working directory and data paths
DATA_DIR = Path.cwd()
DATA_CANDIDATES = [
    Path(
        "/Users/khaledalanjery/Library/CloudStorage/GoogleDrive-khaled@khaledalanjery.com/My Drive/Design Independent Research and Experimentation/DIRE Projects/Aeolian/Aeolian General/Aeolian Github/aeolian-project-repo/msys-project-repo-folder/damage/damage data/damage sites aleppo"
    ),
    DATA_DIR
    / "content/assignments/mapping-systems-assignments/damage/damage data/UNOSAT_CE20130604SYR_Syria_Damage_Assessment_2016_shp",
    DATA_DIR
    / "content/assignments/mapping-systems-assignments/damage/damage data/damage sites aleppo",
    DATA_DIR,
]


def find_data_file(name):
    for base in DATA_CANDIDATES:
        candidate = Path(base) / name
        if candidate.exists():
            return candidate
    return None


damage_points_path = find_data_file("6_Damage_Sites_Aleppo_SDA.shp")
neighborhood_path = find_data_file("7_Aleppo_PercentageDamage_Neighborhood.shp")
residential_damage_path = find_data_file("8_Aleppo_ResidentialPercentageDamage.shp")

# Load source data in EPSG:4326
if damage_points_path is not None:
    gdf_sites = gpd.read_file(damage_points_path)
else:
    gdf_sites = gpd.GeoDataFrame(
        columns=["SiteID", "Notes", "geometry"], geometry="geometry", crs="EPSG:4326"
    )

if neighborhood_path is not None:
    gdf_neighborhoods = gpd.read_file(neighborhood_path)
else:
    gdf_neighborhoods = gpd.GeoDataFrame(
        columns=["Name", "Percent", "geometry"], geometry="geometry", crs="EPSG:4326"
    )

if residential_damage_path is not None:
    gdf_residential = gpd.read_file(residential_damage_path)
else:
    gdf_residential = gpd.GeoDataFrame(
        columns=["ID", "GRIDCODE", "geometry"], geometry="geometry", crs="EPSG:4326"
    )

# Filter to Aleppo historic-district neighborhoods and sites
historic_keywords = [
    "citadel",
    "jalloum",
    "aqabeh",
    "farafira",
    "bayadah",
    "kallasah",
    "sukkari",
    "bab alfaraj",
    "old city",
    "souk",
    "historic district",
]

name_columns = [
    col
    for col in ["Name", "Name_Ar", "Neighbrhd", "StlmtNme"]
    if col in gdf_neighborhoods.columns
]
if len(gdf_neighborhoods) > 0 and name_columns:
    historic_mask = pd.Series(False, index=gdf_neighborhoods.index)
    for col in name_columns:
        values = gdf_neighborhoods[col].fillna("").astype(str).str.lower()
        historic_mask = historic_mask | values.str.contains(
            "|".join(historic_keywords), na=False
        )
    gdf_historic_district = gdf_neighborhoods.loc[historic_mask].copy()
else:
    gdf_historic_district = gdf_neighborhoods.copy()

if (
    len(gdf_historic_district) > 0
    and len(gdf_sites) > 0
    and "geometry" in gdf_sites.columns
    and "geometry" in gdf_historic_district.columns
):
    gdf_sites = gpd.sjoin(
        gdf_sites.to_crs(gdf_historic_district.crs),
        gdf_historic_district[["Name", "Name_Ar", "geometry"]],
        how="inner",
        predicate="within",
    )
    if "index_right" in gdf_sites.columns:
        gdf_sites = gdf_sites.drop(columns=["index_right"])
else:
    site_text_columns = [
        col
        for col in ["Notes", "StlmtNme", "Neighbrhd", "SiteID"]
        if col in gdf_sites.columns
    ]
    if len(gdf_sites) > 0 and site_text_columns:
        combined = gdf_sites[site_text_columns].fillna("").astype(str)
        site_mask = combined.apply(
            lambda row: any(
                keyword in " ".join(row.tolist()).lower()
                for keyword in historic_keywords
            ),
            axis=1,
        )
        gdf_sites = gdf_sites.loc[site_mask].copy()

if (
    len(gdf_historic_district) > 0
    and len(gdf_residential) > 0
    and "geometry" in gdf_residential.columns
):
    gdf_residential = gpd.sjoin(
        gdf_residential.to_crs(gdf_historic_district.crs),
        gdf_historic_district[["geometry"]],
        how="inner",
        predicate="intersects",
    )
    if "index_right" in gdf_residential.columns:
        gdf_residential = gdf_residential.drop(columns=["index_right"])

print(
    "Loaded",
    len(gdf_sites),
    "damage points,",
    len(gdf_historic_district),
    "historic-district neighborhoods, and",
    len(gdf_residential),
    "residential damage features",
)

Loaded 2879 damage points, 8 historic-district neighborhoods, and 20459 residential damage features


In [14]:
import plotly.graph_objects as go
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path

MAPBOX_TOKEN = "pk.eyJ1Ijoia2hhbGVkYWxhbmplcnkiLCJhIjoiY21zOXJtdGE3MHM5NDJ3b2RhZDhlN3RzdiJ9.u-954HDL1cS8ZVODaKr3IQ"

# Load neighborhood boundaries if they were not already created earlier in the notebook
if "gdf_historic_district" not in globals():
    DATA_DIR = Path.cwd()
    DATA_CANDIDATES = [
        Path(
            "/Users/khaledalanjery/Library/CloudStorage/GoogleDrive-khaled@khaledalanjery.com/My Drive/Design Independent Research and Experimentation/DIRE Projects/Aeolian/Aeolian General/Aeolian Github/aeolian-project-repo/msys-project-repo-folder/damage/damage data/damage sites aleppo"
        ),
        DATA_DIR
        / "content/assignments/mapping-systems-assignments/damage/damage data/UNOSAT_CE20130604SYR_Syria_Damage_Assessment_2016_shp",
        DATA_DIR
        / "content/assignments/mapping-systems-assignments/damage/damage data/damage sites aleppo",
        DATA_DIR,
    ]

    def find_data_file(name):
        for base in DATA_CANDIDATES:
            candidate = Path(base) / name
            if candidate.exists():
                return candidate
        return None

    neighborhood_path = find_data_file("7_Aleppo_PercentageDamage_Neighborhood.shp")
    if neighborhood_path is not None:
        gdf_neighborhoods = gpd.read_file(neighborhood_path)
        historic_keywords = [
            "citadel",
            "jalloum",
            "aqabeh",
            "farafira",
            "bayadah",
            "kallasah",
            "sukkari",
            "bab alfaraj",
            "old city",
            "souk",
            "historic district",
        ]
        name_columns = [
            col
            for col in ["Name", "Name_Ar", "Neighbrhd", "StlmtNme"]
            if col in gdf_neighborhoods.columns
        ]
        if len(gdf_neighborhoods) > 0 and name_columns:
            historic_mask = pd.Series(False, index=gdf_neighborhoods.index)
            for col in name_columns:
                values = gdf_neighborhoods[col].fillna("").astype(str).str.lower()
                historic_mask = historic_mask | values.str.contains(
                    "|".join(historic_keywords), na=False
                )
            gdf_historic_district = gdf_neighborhoods.loc[historic_mask].copy()
        else:
            gdf_historic_district = gdf_neighborhoods.copy()
    else:
        gdf_historic_district = gpd.GeoDataFrame(
            columns=["Name", "geometry"], geometry="geometry", crs="EPSG:4326"
        )

# Load the Aleppo damage points if they were not already created earlier in the notebook
if "gdf_sites" not in globals():
    aleppo_path = Path(
        "/Users/khaledalanjery/Library/CloudStorage/GoogleDrive-khaled@khaledalanjery.com/My Drive/Design Independent Research and Experimentation/DIRE Projects/Aeolian/Aeolian General/Aeolian Github/aeolian-project-repo/msys-project-repo-folder/damage/damage data/damage sites aleppo/6_Damage_Sites_Aleppo_SDA.shp"
    )
    if aleppo_path.exists():
        gdf_sites = gpd.read_file(aleppo_path)
    else:
        gdf_sites = gpd.GeoDataFrame(
            columns=["DmgCls_4", "geometry"], geometry="geometry", crs="EPSG:4326"
        )

# Keep only rows that have a usable geometry and damage class
plot_gdf = gdf_sites.dropna(subset=["geometry"]).copy()
if "DmgCls_4" in plot_gdf.columns:
    plot_gdf = plot_gdf.dropna(subset=["DmgCls_4"]).copy()
if getattr(plot_gdf, "crs", None) is not None:
    plot_gdf = plot_gdf.to_crs(epsg=4326)
plot_gdf["lon"] = plot_gdf.geometry.x
plot_gdf["lat"] = plot_gdf.geometry.y
plot_gdf = plot_gdf.dropna(subset=["lon", "lat"])

# Sample for a responsive map while preserving the overall pattern
sample_size = min(1500, len(plot_gdf))
plot_gdf = plot_gdf.sample(n=sample_size, random_state=42).copy()

# Color points by damage class using the requested palette
color_map = {
    "Destroyed": "#ff0000",
    "Severe Damage": "#ff8c00",
    "Moderate Damage": "#ffffff",
    "Medium Moderate Damage": "#ffffff",
    "Low damage": "#ffffff",
    "No damage": "#ffffff",
}

# Create the interactive map
trace_points = go.Scattermapbox(
    lat=plot_gdf["lat"],
    lon=plot_gdf["lon"],
    mode="markers",
    marker=dict(
        size=8,
        color=plot_gdf["DmgCls_4"].map(color_map).fillna("#ffffff"),
        opacity=0.95,
    ),
    text=plot_gdf["DmgCls_4"],
    hovertemplate="<b>%{text}</b><extra></extra>",
    name="Damage class",
)

fig = go.Figure(data=[trace_points])
fig.update_layout(
    mapbox=dict(
        style="mapbox://styles/mapbox/light-v10",
        center=dict(lon=37.16, lat=36.20),
        zoom=10,
        accesstoken=MAPBOX_TOKEN,
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    height=850,
    showlegend=False,
)
fig.show()

In [9]:
from pathlib import Path
import networkx as nx
import geopandas as gpd
import pandas as pd

# Export the network graph as GEXF in the Downloads folder
output_dir = Path.home() / "Downloads"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "aleppo_network.gexf"

citadel_lon = 37.163050615533386
citadel_lat = 36.19949787160008
mosque_lon = 37.1578
mosque_lat = 36.1975


def _project_point(lon, lat):
    return (
        gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs="EPSG:4326")
        .to_crs(epsg=32637)
        .iloc[0]
    )


def _build_point_frame(source_gdf):
    point_frame = source_gdf.dropna(subset=["geometry"]).copy()
    if "lon" not in point_frame.columns or "lat" not in point_frame.columns:
        point_frame = point_frame.to_crs(epsg=4326)
        point_frame["lon"] = point_frame.geometry.x
        point_frame["lat"] = point_frame.geometry.y
    return point_frame


def _nearest_point(point_frame, lon, lat):
    projected = gpd.GeoDataFrame(
        point_frame.copy(),
        geometry=gpd.points_from_xy(point_frame["lon"], point_frame["lat"]),
        crs="EPSG:4326",
    ).to_crs(epsg=32637)
    target = _project_point(lon, lat)
    nearest_idx = projected.geometry.distance(target).idxmin()
    return nearest_idx, projected.loc[nearest_idx, "geometry"], target


# Build the graph from the full filtered Aleppo damage points when available.
if (
    "gdf_sites" in globals()
    and globals()["gdf_sites"] is not None
    and len(globals()["gdf_sites"]) > 0
):
    point_data = _build_point_frame(globals()["gdf_sites"])
elif (
    "plot_gdf" in globals()
    and globals()["plot_gdf"] is not None
    and len(globals()["plot_gdf"]) > 0
):
    point_data = _build_point_frame(globals()["plot_gdf"])
else:
    point_data = gpd.GeoDataFrame(
        columns=["DmgCls_4", "geometry", "lon", "lat"],
        geometry="geometry",
        crs="EPSG:4326",
    )

if len(point_data) > 0:
    graph = nx.Graph()
    point_data_utm = gpd.GeoDataFrame(
        point_data.copy(),
        geometry=gpd.points_from_xy(point_data["lon"], point_data["lat"]),
        crs="EPSG:4326",
    ).to_crs(epsg=32637)

    for idx, row in point_data.iterrows():
        neighborhood = (
            row.get("Neighbrhd") or row.get("StlmtNme") or row.get("Name") or "Unknown"
        )
        graph.add_node(
            idx,
            label=row.get("DmgCls_4", "building"),
            lon=float(row["lon"]),
            lat=float(row["lat"]),
            neighborhood=str(neighborhood),
            damage_class=str(row.get("DmgCls_4", "")),
        )

    graph.add_node(
        "aleppo_citadel",
        label="Aleppo Citadel",
        lon=citadel_lon,
        lat=citadel_lat,
        kind="landmark",
    )
    graph.add_node(
        "al_adiliyah_mosque",
        label="Al-Adiliyah Mosque",
        lon=mosque_lon,
        lat=mosque_lat,
        kind="landmark",
    )

    for neighborhood, nodes in (
        point_data.groupby("Neighbrhd")
        if "Neighbrhd" in point_data.columns
        else [(None, point_data)]
    ):
        node_ids = [node for node in nodes.index if node in graph.nodes]
        if len(node_ids) < 2:
            continue

        ordered_nodes = sorted(
            node_ids,
            key=lambda node_id: (
                graph.nodes[node_id].get("lon", 0),
                graph.nodes[node_id].get("lat", 0),
                str(node_id),
            ),
        )
        for start_node, end_node in zip(ordered_nodes, ordered_nodes[1:]):
            start_geom = point_data_utm.loc[start_node, "geometry"]
            end_geom = point_data_utm.loc[end_node, "geometry"]
            distance_m = float(start_geom.distance(end_geom))
            graph.add_edge(
                start_node,
                end_node,
                edge_type="point_network",
                neighborhood=str(neighborhood),
                weight=distance_m,
                distance_m=distance_m,
            )

    closest_citadel_idx, closest_citadel_geom, citadel_geom = _nearest_point(
        point_data, citadel_lon, citadel_lat
    )
    closest_mosque_idx, closest_mosque_geom, mosque_geom = _nearest_point(
        point_data, mosque_lon, mosque_lat
    )

    graph.add_edge(
        "aleppo_citadel",
        closest_citadel_idx,
        edge_type="anchor",
        weight=0.0,
        distance_m=0.0,
    )
    graph.add_edge(
        "al_adiliyah_mosque",
        closest_mosque_idx,
        edge_type="anchor",
        weight=0.0,
        distance_m=0.0,
    )

    euclidean_distance_m = float(closest_citadel_geom.distance(mosque_geom))
    graph.add_edge(
        closest_citadel_idx,
        "al_adiliyah_mosque",
        edge_type="euclidean",
        label="Closest point to Aleppo Citadel -> Al-Adiliyah Mosque",
        weight=euclidean_distance_m,
        distance_m=euclidean_distance_m,
    )

    try:
        network_distance_m = float(
            nx.shortest_path_length(
                graph,
                "aleppo_citadel",
                "al_adiliyah_mosque",
                weight="distance_m",
            )
        )
    except nx.NetworkXNoPath:
        network_distance_m = float(citadel_geom.distance(mosque_geom))

    graph.add_edge(
        "aleppo_citadel",
        "al_adiliyah_mosque",
        edge_type="network",
        label="Aleppo Citadel -> Al-Adiliyah Mosque",
        weight=network_distance_m,
        distance_m=network_distance_m,
    )

    nx.write_gexf(graph, str(output_path))
    print(
        f"Saved GEXF to {output_path} ({graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges)"
    )
    print(
        f"Closest Citadel point: {closest_citadel_idx}; Euclidean edge: {euclidean_distance_m:.1f} m; "
        f"Network edge: {network_distance_m:.1f} m"
    )
else:
    print(
        "No filtered point data named gdf_sites or plot_gdf was found in the notebook."
    )

Saved GEXF to /Users/khaledalanjery/Downloads/aleppo_network.gexf (2881 nodes, 2875 edges)
Closest Citadel point: 2136; Euclidean edge: 478.8 m; Network edge: 478.8 m


In [4]:
from pathlib import Path
import json

# Export the filtered Aleppo damage points to JSON in the Downloads folder
output_dir = Path.home() / "Downloads"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "aleppo_damage_points.json"

if "plot_gdf" in globals():
    json_data = plot_gdf[["DmgCls_4", "lon", "lat"]].to_dict(orient="records")
    output_path.write_text(json.dumps(json_data, indent=2))
    print(f"Saved JSON to {output_path}")
else:
    print("No GeoDataFrame named plot_gdf was found in the notebook.")

Saved JSON to /Users/khaledalanjery/Downloads/aleppo_damage_points.json


In [5]:
from pathlib import Path

# Export the filtered Aleppo damage points to GeoJSON in the Downloads folder
output_dir = Path.home() / "Downloads"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "aleppo_damage_points.geojson"

if "plot_gdf" in globals():
    plot_gdf[["DmgCls_4", "geometry"]].to_file(output_path, driver="GeoJSON")
    print(f"Saved GeoJSON to {output_path}")
else:
    print("No GeoDataFrame named plot_gdf was found in the notebook.")

Saved GeoJSON to /Users/khaledalanjery/Downloads/aleppo_damage_points.geojson


In [13]:
import plotly.graph_objects as go

# Final map: all filtered points, plus the Aleppo Citadel and Al-Adiliyah Mosque
# with the path-based comparison swapped in the visualization as requested.
if (
    "gdf_sites" in globals()
    and globals()["gdf_sites"] is not None
    and len(globals()["gdf_sites"]) > 0
):
    final_points = gdf_sites.dropna(subset=["geometry"]).copy()
else:
    final_points = gpd.GeoDataFrame(
        columns=["DmgCls_4", "geometry"], geometry="geometry", crs="EPSG:4326"
    )

if getattr(final_points, "crs", None) is not None:
    final_points = final_points.to_crs(epsg=4326)
final_points["lon"] = final_points.geometry.x
final_points["lat"] = final_points.geometry.y

point_color_map = {
    "Destroyed": "#ff0000",
    "Severe Damage": "#ff8c00",
    "Moderate Damage": "#ffffff",
    "Medium Moderate Damage": "#ffffff",
    "Low damage": "#ffffff",
    "No damage": "#ffffff",
}

all_points_trace = go.Scattermapbox(
    lat=final_points["lat"],
    lon=final_points["lon"],
    mode="markers",
    marker=dict(
        size=6,
        color=final_points.get("DmgCls_4", pd.Series(index=final_points.index))
        .map(point_color_map)
        .fillna("#777777"),
        opacity=0.72,
    ),
    text=final_points.get("DmgCls_4", pd.Series(index=final_points.index)).fillna(
        "Unknown"
    ),
    hovertemplate="<b>%{text}</b><extra></extra>",
    name="Damage points",
)

citadel_trace = go.Scattermapbox(
    lat=[citadel_lat],
    lon=[citadel_lon],
    mode="markers+text",
    marker=dict(size=16, color="#111111", symbol="star"),
    text=["Aleppo Citadel"],
    textposition="top center",
    name="Aleppo Citadel",
    hovertemplate="<b>Aleppo Citadel</b><extra></extra>",
)

mosque_trace = go.Scattermapbox(
    lat=[mosque_lat],
    lon=[mosque_lon],
    mode="markers+text",
    marker=dict(size=16, color="#003cff", symbol="star"),
    text=["Al-Adiliyah Mosque"],
    textposition="top center",
    name="Al-Adiliyah Mosque",
    hovertemplate="<b>Al-Adiliyah Mosque</b><extra></extra>",
)

route_lons = []
route_lats = []
if "graph" in globals() and graph is not None:
    try:
        network_route = nx.shortest_path(
            graph, "aleppo_citadel", "al_adiliyah_mosque", weight="distance_m"
        )
        for node_id in network_route:
            route_lons.append(float(graph.nodes[node_id]["lon"]))
            route_lats.append(float(graph.nodes[node_id]["lat"]))
    except (nx.NetworkXNoPath, nx.NodeNotFound, KeyError, TypeError):
        route_lons = [citadel_lon, mosque_lon]
        route_lats = [citadel_lat, mosque_lat]
else:
    route_lons = [citadel_lon, mosque_lon]
    route_lats = [citadel_lat, mosque_lat]

euclidean_trace = go.Scattermapbox(
    lat=route_lats,
    lon=route_lons,
    mode="lines",
    line=dict(color="#00c2ff", width=5),
    name="Euclidean distance",
    hoverinfo="skip",
)

network_trace = go.Scattermapbox(
    lat=[citadel_lat, mosque_lat],
    lon=[citadel_lon, mosque_lon],
    mode="lines",
    line=dict(color="#ff2d55", width=4),
    name="Network distance",
    hoverinfo="skip",
)

fig = go.Figure(
    data=[all_points_trace, network_trace, euclidean_trace, citadel_trace, mosque_trace]
)
fig.update_layout(
    mapbox=dict(
        style="mapbox://styles/mapbox/light-v10",
        center=dict(lon=37.16, lat=36.20),
        zoom=14,
        accesstoken=MAPBOX_TOKEN,
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    height=900,
    showlegend=True,
)
fig.show()